# 使用LLaMA2模型进行微调笔记

## 1. 安装环境所需

In [ ]:
%%capture
%pip install accelerate peft bitsandbytes transformers trl

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig
from trl import SFTTrainer

## 2. 模型配置

### 数据集
- timdettmers/openassistant-guanaco
    - 专门用于训练和评估对话模型，特别是用于提升模型在多轮对话中的表现。

实验使用的是mlabonne/guanaco-llama2-1k，该数据集是timdettmers/openassistant-guanaco的子集，包含1000个对话。

In [ ]:
# Model from Hugging Face hub
base_model = "NousResearch/Llama-2-7b-chat-hf"

# New instruction dataset
guanaco_dataset = "mlabonne/guanaco-llama2-1k"

# Fine-tuned model
new_model = "llama-2-7b-chat-guanaco"

## 3. 加载数据

In [ ]:
dataset = load_dataset(guanaco_dataset, split="train")

## 4. 4位量化配置

In [ ]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

## 5. 加载模型

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

## 6. 加载分词器

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## 7. PEFT参数

In [ ]:
peft_params = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

## 8. 训练参数

In [ ]:
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,  # 每个GPU的batch size
    gradient_accumulation_steps=1,  # 梯度累积
    optim="paged_adamw_32bit",
    save_steps=25,  # 每多少步保存一次模型
    logging_steps=25,  # 每多少步记录一次日志
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,  
    bf16=False,  
    max_grad_norm=0.3,  # 梯度裁剪
    max_steps=-1,  # 最大训练步数
    warmup_ratio=0.03,  # 学习率预热比例
    group_by_length=True,  # 按长度分组
    lr_scheduler_type="constant",  # 学习率衰减策略
    report_to="tensorboard"  # 报告输出，可选择wandb
)

如果想要将数据传至wandb，可以使用以下代码
```
import wandb

# 初始化 wandb
run = wandb.init(
    project="my-project",  # 你的项目名称
    name="my-experiment",  # 你的实验名称
    tags=["tag1", "tag2"],  # 你的实验标签
    # 其他参数...
)
# 在训练结束后
run.finish()
```

在训练大模型时常用的初始化设置
```
import wandb

run = wandb.init(
    project="my-large-model-project",
    name="large-model-run-1",
    config={
        "model": "MyLargeModel",
        "batch_size": 64,
        "learning_rate": 0.001,
        # 其他模型和训练参数...
    },
    notes="Training run for my large model",
    tags=["large-model", "training"],
    resume=True,
    dir="/path/to/my/run/data",
    group="large-model-training",
    job_type="training",
)
```

## 9. 模型微调

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_params,
    dataset_text_field="text",
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_params,
    packing=False,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained(new_model)
trainer.tokenizer.save_pretrained(new_model)

## 10. 评估模型
使用wandb可视化工具，也可以查看模型的训练和评估结果。

In [ ]:
from tensorboard import notebook
log_dir = "results/runs"
notebook.start("--logdir {} --port 4000".format(log_dir))

### 测试结果

In [ ]:
logging.set_verbosity(logging.CRITICAL)  # 取消日志输出

prompt = "Who is Leonardo Da Vinci?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

In [ ]:
prompt = "What is Datacamp Career track?"
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])